# Frame reconstruction between frames (`recon_eval.py`)
Pretrained **E2VID** and **FireNet** reconstruct frame *k* from the events between camera frames *k−1* and *k* (5-bin voxel grid, EVREAL's standard protocol), for the real events and every method in `data_test_evt/<DATASET>/<SPLIT>/`. Nothing is trained.

- **vs real** (`agreement_*`, the main result): each method's reconstruction compared with the reconstruction from the **real recorded events** of the same scene. Same network, same window: the closer, the more realistic the simulated events.
- **vs GT** (`gt_*`, shown only with `SHOW_GT = True`): each reconstruction compared with the **camera frame**. The `real` row is the reference level; simulators were given these frames, so beating `real` here does not mean more realistic events.
- MSE and LPIPS: lower is better; SSIM: higher is better.

Comparison images (rows = networks; columns = real, every method, and GT when `SHOW_GT`) are saved in the run's report folder under `figures/`.

In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
from IPython.display import display, HTML, Image as IPImage

REPO = Path.cwd().resolve()
while not (REPO / "benchmarks" / "recon_eval.py").is_file():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "benchmarks"))
import recon_eval
import recon_figures

REAL, EVTWEEN, HIGHER = recon_eval.REAL, set(recon_eval.EVTWEEN), recon_eval.HIGHER_IS_BETTER
ORDER = {m: i for i, m in enumerate((REAL, *recon_eval.METHODS))}
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)


def latest(base, marker):
    """Newest run folder under base that contains marker (i.e. a finished run)."""
    runs = sorted(p for p in Path(base).glob("*") if (p / marker).is_file())
    return runs[-1] if runs else None


def resolve(run, base, marker):
    if run is None:
        return None
    if run == "latest":
        found = latest(base, marker)
        if found is None:
            print(f"No finished run under {base}")
        return found
    path = Path(run) if Path(run).is_absolute() else REPO / run
    if not (path / marker).is_file():
        raise FileNotFoundError(f"{path} has no {marker} (not a finished run?)")
    return path


def show_figure(path):
    if path is None:
        print("   (no saved frames for this one)")
        return
    print("   saved:", path.relative_to(REPO) if path.is_relative_to(REPO) else path)
    display(IPImage(filename=str(path)))

# ---- settings -------------------------------------------------------------
DATASET = "eds"            # folder name under data_test_evt/ (case-sensitive)
SPLIT = "test"
RUN_DIR = "latest"         # "latest" = newest finished run of DATASET; a report folder; or None to start a new run below
SHOW_GT = True             # False: only the comparison with real events (tables, tests and images)
METHODS = None             # for a new run: None = every method found, or a list
DEVICES = ["cuda:0", "cuda:0"]

run_dir = resolve(RUN_DIR, REPO / "reports" / "recon" / DATASET / SPLIT, "summary.csv") if RUN_DIR else None
print("run:", run_dir)

## Run (only for a new run)
Long runs are safer in a terminal (`tmux`); the command is printed. Skip this cell to only view `RUN_DIR`.

In [ ]:
if RUN_DIR is None:
    argv = [DATASET, "--split", SPLIT, "--devices", *DEVICES] + (["--methods", *METHODS] if METHODS else [])
    print("python benchmarks/recon_eval.py " + " ".join(argv))
    run_dir = recon_eval.main(argv)
else:
    print("Viewing", run_dir, "(set RUN_DIR = None to start a new run)")

In [ ]:
recon_eval.summarize(run_dir)
summary = pd.read_csv(run_dir / "summary.csv")
config = json.loads((run_dir / "config.json").read_text())
print("methods:", config["methods"])


def table(kind):
    cols = {}
    for m in recon_eval.METRICS:
        c = f"{kind}_{m}"
        cols[f"{m.upper()} {'↑' if HIGHER[m] else '↓'}"] = summary.apply(
            lambda r: f"{r[c+'_mean']:.4f} ± {r[c+'_std']:.4f} ({int(r[c+'_rank'])})" if pd.notna(r[c + "_mean"]) else "",
            axis=1)
    out = pd.concat([summary[["network", "method", "sequences", "frames"]], pd.DataFrame(cols)], axis=1)
    if kind == "agreement":
        out = out[out.method != REAL]
    return out.set_index(["network", "method"])

## Main result: vs real
Mean ± std over sequences (each sequence weighted equally), rank in brackets.

In [ ]:
display(table("agreement"))

## vs GT (camera frames)
Only with `SHOW_GT = True`. Read next to the `real` row.

In [ ]:
if SHOW_GT:
    display(table("gt"))
else:
    print("SHOW_GT = False: skipped")

## Sanity check of the inputs
Event count, rate, ON fraction and empty windows per source; `status = missing` = no output for that sequence.

In [ ]:
sanity = pd.read_csv(run_dir / "sanity.csv")
display(sanity.groupby("method", sort=False)[["events", "rate_hz", "on_fraction", "empty_windows"]].mean())
display(sanity[sanity.status != "ok"])

## Best Evtween variant vs best baseline
Paired Wilcoxon test over sequences. With fewer than 6 sequences the two-sided p-value cannot go below 0.05.

In [ ]:
wx = pd.read_csv(run_dir / "wilcoxon.csv")
display(wx if SHOW_GT else wx[wx["metric"].str.startswith("agreement")])

## Comparison images of the saved frames
The frames saved by the run (40 per sequence with `--save hybrid`, chosen as highest/lowest PSNR, most motion, evenly spread). Each image is saved to `figures/` in the report folder.

In [ ]:
FIGURE_SEQUENCE = None       # None = first sequence
FIGURE_CATEGORY = "high_psnr"  # "high_psnr" | "low_psnr" | "high_motion" | "temporal" | None = any
FIGURE_LIMIT = 3

sel_file = run_dir / "selection.csv"
if sel_file.is_file():
    selection = pd.read_csv(sel_file)
    display(selection.groupby(["sequence", "category"]).size().unstack(fill_value=0))
else:  # --save samples/all: whatever frames were saved
    root = recon_figures.recon_root(run_dir, "frames")
    selection = pd.DataFrame([dict(sequence=p.parent.name, frame=int(p.stem), category="saved")
                              for p in sorted((root / config["networks"][0] / REAL).glob("*/*.png"))])
seq = FIGURE_SEQUENCE or (selection["sequence"].iloc[0] if not selection.empty else None)
pick = selection[(selection["sequence"] == seq) &
                 ((selection["category"] == FIGURE_CATEGORY) if FIGURE_CATEGORY and "saved" not in set(selection.category)
                  else True)].head(FIGURE_LIMIT)
for _, row in pick.iterrows():
    print(f"{seq} frame {row.frame} ({row.category})")
    show_figure(recon_figures.frame_figure(run_dir, seq, row.frame, f"{row.category}_{seq}_{int(row.frame):06d}",
                                           show_gt=SHOW_GT, title=f"({row.category})"))